# 🧭 PCA — Principal Component Analysis
**ISLP Chapter 12 · Pattern Recognition for the Rest of Us**

> When you have 50 features, PCA finds the 2-3 directions that capture the most variation — compressing information without (much) loss. It's the most widely-used dimensionality reduction technique in data science.

### What you'll learn
- What a principal component is geometrically
- Loadings, scores, and the scree plot
- Proportion of variance explained
- Using PCA for visualization, preprocessing, and noise reduction
- Biplot — variables and observations in the same space

### Dataset: USArrests + Digits (handwritten numbers)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
import scipy.stats as stats

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: USArrests

In [ ]:
import pandas as pd
arrests = pd.read_csv(f'{DATA_DIR}/USArrests.csv', index_col=0)
print(f"USArrests: {arrests.shape}  |  States: {list(arrests.index[:5])}...")
arrests.head()

## 📐 Part 1 — What PCA Does

PCA finds a new coordinate system where:
- **PC1** is the direction of maximum variance in the data
- **PC2** is the direction of maximum *remaining* variance, orthogonal to PC1
- Each subsequent PC captures the most remaining variance

Mathematically, PCA solves: find unit vector φ₁ that maximizes Var(Xφ₁).

The solution: φ₁ is the first **eigenvector** of the covariance matrix X^TX.

**Key outputs:**
- **Loadings** (φ) — how much each original variable contributes to each PC
- **Scores** (Z = Xφ) — the new coordinates of each observation
- **Variance explained** — how much information each PC captures

In [ ]:
# Apply PCA to USArrests
scaler = StandardScaler()
X_scaled = scaler.fit_transform(arrests)

pca = PCA()
scores = pca.fit_transform(X_scaled)

print("=== PCA Results ===\n")
loadings = pd.DataFrame(pca.components_.T,
                         index=arrests.columns,
                         columns=[f'PC{i+1}' for i in range(4)])
print("Loadings (how each feature contributes to each PC):")
print(loadings.round(3))

print("\nVariance explained:")
for i, var in enumerate(pca.explained_variance_ratio_):
    cumvar = pca.explained_variance_ratio_[:i+1].sum()
    print(f"  PC{i+1}: {var:.3f} ({var*100:.1f}%)  |  Cumulative: {cumvar*100:.1f}%")

In [ ]:
# Biplot — show states and variables together
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Scree plot
axes[0].bar(range(1,5), pca.explained_variance_ratio_*100,
            color='#1e5fa8', edgecolor='white', alpha=0.8, label='Individual')
axes[0].plot(range(1,5), np.cumsum(pca.explained_variance_ratio_)*100,
             'o-', color='#e85d20', lw=2, markersize=8, label='Cumulative')
axes[0].axhline(80, color='#888', lw=1, ls='--', alpha=0.6, label='80% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot — PC1+PC2 explain 86.8% of variance')
axes[0].legend()
axes[0].set_xticks([1,2,3,4])

# Right: Biplot
scale_factor = 3
axes[1].scatter(scores[:,0], scores[:,1], color='#888', alpha=0.7, s=30, zorder=3)
for i, state in enumerate(arrests.index):
    axes[1].annotate(state, (scores[i,0], scores[i,1]), fontsize=6.5, alpha=0.8)

for j, feat in enumerate(arrests.columns):
    axes[1].annotate('', xy=(loadings.iloc[j,0]*scale_factor, loadings.iloc[j,1]*scale_factor),
                     xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#e85d20', lw=2))
    axes[1].text(loadings.iloc[j,0]*scale_factor*1.1, loadings.iloc[j,1]*scale_factor*1.1,
                 feat, color='#e85d20', fontweight='bold', fontsize=10)

axes[1].axhline(0, color='black', lw=0.5, ls='--', alpha=0.3)
axes[1].axvline(0, color='black', lw=0.5, ls='--', alpha=0.3)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[1].set_title('Biplot — States and Crime Variables in PC Space')

plt.tight_layout()
plt.show()
print("\n📌 PC1: all crime variables point right → 'overall crime' axis")
print("   PC2: UrbanPop points up, crime down → 'urban vs rural' axis")
print("   States far right (FL, NV) = high crime. Top right (CA) = high crime + urban")

In [ ]:
# PCA on high-dimensional data: handwritten digits
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
print(f"Digits: {X_dig.shape} — {X_dig.shape[0]} images, {X_dig.shape[1]} pixels each")

pca_dig = PCA(n_components=50)
X_reduced = pca_dig.fit_transform(X_dig)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scree
cumvar = np.cumsum(pca_dig.explained_variance_ratio_)
axes[0].plot(range(1,51), cumvar*100, color='#1e5fa8', lw=2)
n_90 = np.argmax(cumvar >= 0.90) + 1
axes[0].axvline(n_90, color='#e85d20', lw=1.5, ls='--', label=f'{n_90} PCs → 90% variance')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Variance Explained (%)')
axes[0].set_title('Digits: How Many PCs Needed?')
axes[0].legend()

# 2D embedding
colors_dig = plt.cm.tab10(np.linspace(0,1,10))
for digit in range(10):
    mask = y_dig==digit
    axes[1].scatter(X_reduced[mask,0], X_reduced[mask,1],
                    color=colors_dig[digit], alpha=0.5, s=10, label=str(digit))
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('Digits in 2D PCA Space')
axes[1].legend(title='Digit', fontsize=7, ncol=2)

# Reconstruction quality
pca_small = PCA(n_components=n_90).fit(X_dig)
X_recon = pca_small.inverse_transform(pca_small.transform(X_dig))
sample_idx = 42
axes[2].imshow(X_recon[sample_idx].reshape(8,8), cmap='gray_r')
axes[2].set_title(f'Digit {y_dig[sample_idx]} reconstructed\nfrom {n_90} PCs (vs 64 pixels)')
axes[2].axis('off')

plt.suptitle('PCA on Handwritten Digits (64 pixels → fewer dimensions)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print(f"\n📌 {n_90} PCs capture 90% of variance in 64-pixel images")
print(f"   Compression ratio: {64/n_90:.1f}x — and the digit is still recognizable!")

In [ ]:
answers = {
    "q1": "",  # What does the first principal component maximize?
    "q2": "",  # What do loadings tell you about a principal component?
    "q3": "",  # Why must features be standardized before PCA?
    "q4": "",  # You have 100 features. The first 5 PCs explain 85% of variance. What would you do next?
    "q5": "",  # What is one limitation of PCA for classification tasks?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")